In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata

# Private repo: store a fine-grained PAT with read access to ozogxyz/fovea
# in Colab Secrets as GH_PAT. fovea pulls pytorch_pretrained_vit as a dep.
GH_PAT = userdata.get("GH_PAT")
!pip install -q "git+https://{GH_PAT}@github.com/ozogxyz/fovea.git"


In [ ]:
import os
import shutil

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision.transforms import Compose as TorchvisionCompose
from pytorch_pretrained_vit import ViT
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import log_loss

from fovea import LoRA_ViT, ColorJitterCV, RandomGaussianBlur, RandomHorizontalFlip


In [ ]:
ARCHIVE = "/content/drive/MyDrive/datadriven/competition_VfIpjyh.zip"
DATA_DIR = "/content/data"

RANK = 8
NUM_CLASSES = 8
FRAC = 1.0
TEST_SIZE = 0.25
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
SEED = 1
AUGMENT = True
NORM_MEAN = 0.5
NORM_STD = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if "A100" in gpu_name else 32
NUM_WORKERS = os.cpu_count() or 2
print(f"device: {device} ({gpu_name}), batch_size: {BATCH_SIZE}, workers: {NUM_WORKERS}")

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
shutil.unpack_archive(ARCHIVE, DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
def augment(sample):
    return TorchvisionCompose(
        [
            ColorJitterCV(brightness=0.8, contrast=0.1, gamma=0.2, temp=0.8, p=0.75),
            RandomGaussianBlur(),
            RandomHorizontalFlip(),
        ]
    )(sample)


In [ ]:
backbone = ViT("B_16", pretrained=True)
IMG_SIZE = backbone.image_size
IMG_SIZE = IMG_SIZE[0] if isinstance(IMG_SIZE, (tuple, list)) else IMG_SIZE
print(f"image size: {IMG_SIZE}")

In [ ]:
os.chdir(DATA_DIR)
train_features = pd.read_csv("train_features.csv", index_col="id")
test_features = pd.read_csv("test_features.csv", index_col="id")
train_labels = pd.read_csv("train_labels.csv", index_col="id")
species_labels = sorted(train_labels.columns.unique())

In [ ]:
y = train_labels.sample(frac=FRAC, random_state=SEED)
x = train_features.loc[y.index].filepath.to_frame()
sites = train_features.loc[y.index, "site"]
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, eval_idx = next(gss.split(x, y, groups=sites))
x_train, x_eval = x.iloc[train_idx], x.iloc[eval_idx]
y_train, y_eval = y.iloc[train_idx], y.iloc[eval_idx]

In [ ]:
class ImagesDataset(Dataset):
    def __init__(self, x_df, y_df=None, mode="eval"):
        self.data = x_df
        self.label = y_df
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = cv2.imread(self.data.iloc[idx]["filepath"])  # BGR uint8
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        sample = {"image": img}
        if self.mode == "train":
            sample = augment(sample)
        img = cv2.cvtColor(np.ascontiguousarray(sample["image"]), cv2.COLOR_BGR2RGB)
        img = img.transpose(2, 0, 1)
        img = torch.from_numpy(img.copy()).float() / 255.0
        img = (img - NORM_MEAN) / NORM_STD
        out = {"image_id": self.data.index[idx], "image": img}
        if self.label is not None:
            out["label"] = torch.tensor(self.label.iloc[idx].values, dtype=torch.float)
        return out

In [ ]:
train_dataset = ImagesDataset(x_train, y_train, mode="train" if AUGMENT else "eval")
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
eval_dataset = ImagesDataset(x_eval, y_eval, mode="eval")
eval_dataloader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
model = LoRA_ViT(backbone, r=RANK, num_classes=NUM_CLASSES).to(device)
counts = y_train[species_labels].sum().values
weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
def evaluate():
    model.eval()
    parts = []
    with torch.no_grad():
        for batch in eval_dataloader:
            preds = nn.functional.softmax(model(batch["image"].to(device)), dim=1)
            parts.append(pd.DataFrame(preds.cpu().numpy(), index=batch["image_id"], columns=species_labels))
    preds_df = pd.concat(parts).loc[y_eval.index]
    ll = log_loss(y_eval.idxmax(axis=1), preds_df[species_labels], labels=species_labels)
    return ll, preds_df

In [ ]:
best_ll = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch in tqdm(train_dataloader, total=len(train_dataloader)):
        optimizer.zero_grad()
        loss = criterion(model(batch["image"].to(device)), batch["label"].to(device))
        loss.backward()
        optimizer.step()
    ll, _ = evaluate()
    print(f"epoch {epoch:2d}  train_loss {loss.item():.4f}  eval_logloss {ll:.4f}")
    if ll < best_ll:
        best_ll = ll
        torch.save(model.state_dict(), "best.pth")
print(f"best eval log loss: {best_ll:.4f}")

In [ ]:
model.load_state_dict(torch.load("best.pth"))
_, eval_preds_df = evaluate()
eval_true = y_eval.idxmax(axis=1)
eval_predictions = eval_preds_df.idxmax(axis=1)
print(f"best eval accuracy: {(eval_predictions == eval_true).mean():.4f}")
print("per-class accuracy:")
for sp in species_labels:
    mask = eval_true == sp
    if mask.sum():
        print(f"  {sp:18s} {(eval_predictions[mask] == eval_true[mask]).mean():.3f}  (n={mask.sum()})")